# Urea — 4-DMAB / foldscope yellow area (Anitha 2022)

Biosensor path: oilcake + 4-DMAB → yellow. Camera / foldscope measures yellow %.
Ridge per cake maps area → urea g/kg. BIS max **1%** (10 g/kg).


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("../..").resolve()
if not (ROOT / "smartfeed").exists():
    ROOT = Path(".").resolve()
    while ROOT != ROOT.parent and not (ROOT / "smartfeed").exists():
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data" / "processed"
print("ROOT", ROOT)
print("DATA", DATA)


ROOT /Users/sayangarai/Documents/Programming/Projects/SIH_PS_260111
DATA /Users/sayangarai/Documents/Programming/Projects/SIH_PS_260111/data/processed


In [2]:
from smartfeed.urea import assess_urea, fit_urea_model, load_anitha

df = load_anitha()
print(df)
bundle = fit_urea_model()
print("cakes", [k for k in bundle["models"] if k != "__global__"])
print(assess_urea(7.0, "groundnut cake", bundle))
print(assess_urea(60.0, "soybean meal", bundle))


                      oilseed_cake  urea_g_per_kg  yellow_area_pct_mean  \
0                    Soyabean meal              1                  6.66   
1                    Soyabean meal              5                 29.99   
2                    Soyabean meal             10                 59.58   
3                   Groundnut cake              1                  7.35   
4                   Groundnut cake              5                 32.35   
5                   Groundnut cake             10                 63.60   
6  Un-decorticated cottonseed cake              1                  8.19   
7  Un-decorticated cottonseed cake              5                 34.00   
8  Un-decorticated cottonseed cake             10                 67.35   

   yellow_area_pct_se  
0                0.48  
1                0.94  
2                0.64  
3                0.50  
4                0.95  
5                0.98  
6                0.84  
7                0.79  
8                0.89  
cakes ['G

In [3]:
from sklearn.metrics import mean_absolute_error

df = load_anitha()
bundle = fit_urea_model()
pred = []
for r in df.itertuples(index=False):
    pred.append(assess_urea(r.yellow_area_pct_mean, r.oilseed_cake, bundle)["urea_g_per_kg"])
mae = mean_absolute_error(df["urea_g_per_kg"], pred)
print("in-sample MAE g/kg (3-point curves, tiny n)", round(mae, 3))
art = Path("artifacts")
art.mkdir(exist_ok=True)
(art / "metrics.json").write_text(
    json.dumps({"model": "Ridge yellow_area -> urea_g_per_kg", "n": int(len(df)), "in_sample_mae_g_per_kg": mae, "bis_max_pct": 1.0}, indent=2)
)
print("saved artifacts")


in-sample MAE g/kg (3-point curves, tiny n) 0.016
saved artifacts
